# 相対座標でGNNを学習する
### 絶対座標で行った際には絶対位置を記憶してしまう様子が見られたので，それを修正したい
<p>絶対座標から相対座標にする．具体的には構造の中のランダムな点を選択→そのn近傍をinputとし，最初に選択した点の欠陥の有無や層を推論する．</p>

# ライブラリのインポート

In [1]:
import os
import re
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv, LayerNorm
from torch_geometric.data import Data, Batch
from torch_geometric.loader import NeighborLoader
import datetime
from sklearn.metrics import f1_score
import pandas as pd
import warnings
import json
from tqdm.auto import tqdm
from Loss.focal_loss import FocalLoss
from sklearn.metrics import f1_score, recall_score, precision_score


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/home/asano/anaconda3/envs/pyg/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/home/asano/anaconda3/envs/pyg/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/home/asano/anaconda3/envs/pyg/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/home/asano/anaconda3/envs/pyg/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, in launch_in

## 自作モジュール

In [2]:
# ==============================================================================
# ユーティリティ
# ==============================================================================
def extract_layer_block(file_name):
    try:
        m = re.search(r'L(\d+)_B(\d+)_el(\d+)_H8_W8', file_name)
        layer = int(m.group(1))
        block = int(m.group(2))
        el = int(m.group(3))
        return (layer, block, el)
    except AttributeError:
        print(f"Invalid file name format: {file_name}")
        return None


def create_data_label_pairs(data_files, label_files):
    data_label_pairs = {}
    unmatched_labels = set(label_files)
    no_label_counter = 0

    for data_file in data_files:
        key = extract_layer_block(data_file)
        if key is not None:
            data_label_pairs[key] = {"data": data_file}

    for label_file in label_files:
        key = extract_layer_block(label_file)
        if key is not None and key in data_label_pairs:
            if label_file.endswith("_19label.npy"):
                data_label_pairs[key]["label"] = label_file
                unmatched_labels.discard(label_file)

    for _, v in data_label_pairs.items():
        if "label" not in v and no_label_counter < 3:
            print(f"No label found for data file: {v['data']}")
            no_label_counter += 1

    valid_pairs = [
        (v["data"], v["label"])
        for _, v in data_label_pairs.items()
        if "label" in v
    ]

    print(f"{len(valid_pairs)} matched pairs created. {len(unmatched_labels)} labels were unused in this split.")
    return valid_pairs

def compute_class_weights(labels, num_classes=19):
    class_counts = np.bincount(labels, minlength=num_classes)
    class_counts = np.maximum(class_counts, 1)
    class_weights = 1.0 / class_counts
    class_weights = class_weights / class_weights.mean()
    return torch.tensor(class_weights, dtype=torch.float)


def prepare_data(pairs, normalized_data_folder, label_data_folder,
                 x_coords, y_coords, z_coords, edge_index):
    data_list = []
    labels = []
    pair_counter = 0

    for data_file, label_file in pairs:
        data_file_path = os.path.join(normalized_data_folder, data_file)
        label_file_path = os.path.join(label_data_folder, label_file)

        if not os.path.exists(data_file_path) or not os.path.exists(label_file_path):
            continue

        try:
            values = np.load(data_file_path)[:13942]
            label = np.load(label_file_path)[:13942]
        except Exception as e:
            print(f"データ読み込みエラー: {e}")
            continue

        try:
            node_features = np.vstack((x_coords, y_coords, z_coords, values)).T
            x = torch.tensor(node_features, dtype=torch.float)
            y = torch.argmax(torch.tensor(label, dtype=torch.float), dim=1).long()
        except Exception as e:
            print(f"特徴量作成エラー: {e}")
            continue

        data = Data(x=x, edge_index=edge_index, y=y)
        data_list.append(data)
        labels.extend(y.tolist())

        if pair_counter < 1:
            print(f"Pair {pair_counter + 1}:")
            print(f"Data file: {data_file_path}")
            print(f"Label file: {label_file_path}")
            print(f"x shape: {x.shape}, y shape: {y.shape}")
            pair_counter += 1

    if len(labels) == 0:
        print("ラベルが存在しません。クラス重みの計算に失敗しました。")
        return None, None

    class_weights = compute_class_weights(np.array(labels), num_classes=19)
    return data_list, class_weights


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## GNNの関数

In [3]:
# ==============================================================================
# モデル
# ==============================================================================
def initialize_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)


def edge_dropout(edge_index, drop_prob=0.0001):
    if drop_prob == 0:
        return edge_index
    mask = torch.rand(edge_index.size(1), device=edge_index.device) > drop_prob
    return edge_index[:, mask]


class DeepGATModel(torch.nn.Module):
    def __init__(self, hidden_channels=64, num_classes=19, num_layers=5, heads=4):
        super().__init__()
        self.num_layers = num_layers

        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        self.projs = nn.ModuleList()

        # --- first layer ---
        self.convs.append(GATv2Conv(4, hidden_channels, heads=heads, concat=True))
        self.norms.append(nn.BatchNorm1d(hidden_channels * heads))
        self.projs.append(nn.Linear(4, hidden_channels * heads))

        # --- middle layers ---
        for _ in range(num_layers - 2):
            self.convs.append(
                GATv2Conv(hidden_channels * heads, hidden_channels, heads=heads, concat=True)
            )
            self.norms.append(nn.BatchNorm1d(hidden_channels * heads))
            self.projs.append(nn.Identity())

        # --- last layer ---
        self.convs.append(
            GATv2Conv(hidden_channels * heads, hidden_channels, heads=heads, concat=True)
        )
        self.norms.append(nn.BatchNorm1d(hidden_channels * heads))
        self.projs.append(nn.Identity())

        self.fc = nn.Linear(hidden_channels * heads, num_classes)
        self.dropout = nn.Dropout(p=0.05)

        self.apply(initialize_weights)

    def forward(self, x, edge_index, target_indices=None):
        if self.training:
            edge_index = edge_dropout(edge_index, drop_prob=0.0001)

        for i in range(self.num_layers):
            x_residual = x

            x = self.convs[i](x, edge_index)
            x = self.norms[i](x)
            x = F.relu(x)
            x = self.dropout(x)

            if x_residual.size(1) != x.size(1):
                x_residual = self.projs[i](x_residual)

            x = x + x_residual

        if target_indices is not None:
            x = x[target_indices]

        x = self.fc(x)
        x = F.softmax(x, dim=-1)
        return x

Task was destroyed but it is pending!
task: <Task pending name='Task-36' coro=<_async_in_context.<locals>.run_in_context_pre311() running at /home/asano/anaconda3/envs/pyg/lib/python3.10/site-packages/ipykernel/utils.py:76> cb=[ZMQStream._run_callback.<locals>._log_error() at /home/asano/anaconda3/envs/pyg/lib/python3.10/site-packages/zmq/eventloop/zmqstream.py:563]>
/home/asano/anaconda3/envs/pyg/lib/python3.10/asyncio/base_events.py:674: RuntimeWarning: coroutine '_async_in_context.<locals>.run_in_context_pre311' was never awaited
  self._ready.clear()


## パラメータの設定

In [4]:
warnings.filterwarnings("ignore")

print(f"Using device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

experiment_name = "kfold_dynamic_5passing"
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

seed = 42

set_seed(seed=seed)

hidden_channels = 64
learning_rate = 0.001
node_batch_size = 256
val_batch_size = 2048          # 8192 -> 2048 に削減
epochs = 400
weight_decay = 1e-4
patience = 30
gamma = 2.0
n_splits = 5

num_neighbors = [-1,-1,-1,-1,-1]

# 1 epoch において使用するデータ点の割合
# defect→欠陥の箇所のデータ
# bg→欠陥がない箇所のデータ
train_defect_sampling = 0.05
train_bg_sampling = 0.05
train_num_workers = 0

# validation高速化
val_defect_sampling = 0.05     # 0.10 -> 0.05
val_bg_sampling = 0.05         # 0.10 -> 0.05
eval_every = 1                 # 5 epochごとだけ validation
val_num_workers = 0            # 不安定なら 0 に戻す

# 時間がかかるので一旦fold 1のみ
run_only_fold1 = True


label_data_folder = "/home/asano/asano_handover/npy_datas_H8_W8_19_label"
x_coords_path = "/home/asano/asano_handover/npy_datas_H8_W8_xyz_2layer/hole_x_2layer_normalized.npy"
y_coords_path = "/home/asano/asano_handover/npy_datas_H8_W8_xyz_2layer/hole_y_2layer_normalized.npy"
z_coords_path = "/home/asano/asano_handover/npy_datas_H8_W8_xyz_2layer/hole_z_2layer_normalized.npy"
edge_path = "/home/asano/asano_handover/npy_datas_H8_W8_xyz_2layer/hole_edges_2layer_bidirectional.npy"

# noise intensity
intensities = [0.05,0.3,0.7]
dataset_dir_dict = {
    f"wn_{intensity}": f"npy_datas_H8_W8_Region1_21_kfold_wn_{intensity}"
    for intensity in intensities
}

output_dir_base = f"/home/asano/asano_handover/gnn_outputs/{experiment_name}/"
output_dir_template_dict = {}

for dataset_key, dataset_value in dataset_dir_dict.items():
    output_dir_csv = os.path.join(output_dir_base, dataset_key, "predict_csv", "predict_csv.csv")
    output_dir_info = os.path.join(output_dir_base, dataset_key, "training_info")
    output_dir_pth_template = os.path.join(
        output_dir_base, dataset_key, "pth",
        "{model_name}_" + experiment_name + "_best_model_fold_{fold}.pth"
    )

    output_dir_template_dict[f"{dataset_key}_csv"] = output_dir_csv
    output_dir_template_dict[f"{dataset_key}_info"] = output_dir_info
    output_dir_template_dict[f"{dataset_key}_pth_template"] = output_dir_pth_template

    os.makedirs(output_dir_info, exist_ok=True)
    os.makedirs(os.path.join(output_dir_base, dataset_key, "pth"), exist_ok=True)
    os.makedirs(os.path.join(output_dir_base, dataset_key, "predict_csv"), exist_ok=True)

    params = {
        "hyperparameters": {
            "model": "DeepGATModel_5Layers_Dynamic",
            "epochs": epochs,
            "num_neighbors": num_neighbors,
            "sampling_strategy": {
                "train_defect": train_defect_sampling,
                "train_bg": train_bg_sampling,
                "val_defect": val_defect_sampling,
                "val_bg": val_bg_sampling
            },
            "batch_size": {"train": node_batch_size, "val": val_batch_size},
            "disjoint": True,
            "run_only_fold1": run_only_fold1,
            "eval_every": eval_every,
            "val_num_workers": val_num_workers
        }
    }
    with open(os.path.join(output_dir_info, "training_params.json"), "w") as f:
        json.dump(params, f, indent=4)



Using device: NVIDIA GeForce RTX 3090


## Mainの処理

In [ ]:
for dataset_key, dataset_value in dataset_dir_dict.items():

    kfold_base_folder = f"/home/asano/asano_handover/{dataset_value}"
    output_dir_info = output_dir_template_dict[f"{dataset_key}_info"]
    pth_dir_template = output_dir_template_dict[f"{dataset_key}_pth_template"]
    output_dir_csv = output_dir_template_dict[f"{dataset_key}_csv"]

    if not os.path.isdir(kfold_base_folder):
        print(f"Directory not found: {kfold_base_folder}")
        continue

    # ノードとエッジ
    x_coords = np.load(x_coords_path)
    y_coords = np.load(y_coords_path)
    z_coords = np.load(z_coords_path)
    edges = np.load(edge_path)
    edge_index = torch.tensor(edges.T, dtype=torch.long)

    all_fold_metrics = []
    folds_to_run = [0] if run_only_fold1 else list(range(n_splits))

    for fold in folds_to_run:
        print(f"Starting Fold {fold + 1} / {n_splits} for {dataset_key}")

        train_folder = os.path.join(kfold_base_folder, f"fold_{fold}", "train")
        val_folder = os.path.join(kfold_base_folder, f"fold_{fold}", "validation")

        if not os.path.isdir(train_folder):
            continue

        train_data_files = [f for f in os.listdir(train_folder) if f.endswith(".npy")]
        val_data_files = [f for f in os.listdir(val_folder) if f.endswith(".npy")]
        label_files = [f for f in os.listdir(label_data_folder) if f.endswith("_19label.npy")]

        train_pairs = create_data_label_pairs(train_data_files, label_files)
        val_pairs = create_data_label_pairs(val_data_files, label_files)

        train_dataset_list, class_weights = prepare_data(
            train_pairs, train_folder, label_data_folder,
            x_coords, y_coords, z_coords, edge_index
        )
        val_dataset_list, _ = prepare_data(
            val_pairs, val_folder, label_data_folder,
            x_coords, y_coords, z_coords, edge_index
        )

        if train_dataset_list is None or val_dataset_list is None:
            print("Dataset preparation failed.")
            continue

        # １つの巨大なグラフを作成する
        train_data_batch = Batch.from_data_list(train_dataset_list)
        train_data_combined = Data(
            x=train_data_batch.x,
            edge_index=train_data_batch.edge_index,
            y=train_data_batch.y
        )

        val_data_batch = Batch.from_data_list(val_dataset_list)
        val_data_combined = Data(
            x=val_data_batch.x,
            edge_index=val_data_batch.edge_index,
            y=val_data_batch.y
        )

        y_train = train_data_combined.y
        y_val = val_data_combined.y

        val_defect_mask = (y_val != 0) & (torch.rand(y_val.size(0)) < val_defect_sampling)
        val_bg_mask = (y_val == 0) & (torch.rand(y_val.size(0)) < val_bg_sampling)
        val_input_mask = val_defect_mask | val_bg_mask

        if val_input_mask.sum().item() == 0:
            print("Validation input mask is empty. Skip this fold.")
            continue

        print(f"[Val Sampling] Used: {val_input_mask.sum().item()} / {y_val.size(0)}")

        # neighbor loaderを定義する
        val_loader = NeighborLoader(
            val_data_combined,
            input_nodes=val_input_mask,
            num_neighbors=num_neighbors,
            batch_size=val_batch_size,
            shuffle=False,
            disjoint=True,
            num_workers=val_num_workers,
            persistent_workers=(val_num_workers > 0)
        )

        model = DeepGATModel(
            hidden_channels=hidden_channels,
            num_classes=19,
            num_layers=len(num_neighbors),
            heads=4
        ).to(device)

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=learning_rate,
            weight_decay=weight_decay
        )

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=10, verbose=True
        )

        cw_tensor = class_weights.to(device) if class_weights is not None else None
        loss_fn = FocalLoss(weights=cw_tensor, gamma=gamma, reduction="mean").to(device)

        best_val_loss = float("inf")
        counter = 0

        train_loss_hist, val_loss_hist, val_acc_hist = [], [], []
        f1_hist, recall_hist, precision_hist = [], [], []
        f1_weighted_hist, recall_weighted_hist, precision_weighted_hist = [], [], []
        
        structure_ok_train = None
        structure_ok_val = None

        epoch_bar = tqdm(range(1, epochs + 1), desc=f"Fold {fold+1}", position=0, leave=True)
        for epoch in epoch_bar:
            
            # 全体のデータセットの中からランダムにデータを選択する
            # 欠陥あり/なし　ごとで個別に選択
            train_defect_mask = (y_train != 0) & (torch.rand(y_train.size(0)) < train_defect_sampling)
            train_bg_mask = (y_train == 0) & (torch.rand(y_train.size(0)) < train_bg_sampling)
            train_input_mask = train_defect_mask | train_bg_mask

            train_loader = NeighborLoader(
                train_data_combined,
                input_nodes=train_input_mask,
                num_neighbors=num_neighbors,
                batch_size=node_batch_size,
                shuffle=True,
                disjoint=True,
                num_workers=train_num_workers,
                persistent_workers=False
            )

            model.train()
            total_loss = 0.0
            total_batches = 0

            for batch_idx, batch in enumerate(tqdm(train_loader, desc="Train", leave=False, position=1)):
                batch = batch.to(device)
                optimizer.zero_grad()

                batch_size = batch.batch_size
                seed_indices = torch.arange(batch_size, device=device)

                assert seed_indices.numel() == batch_size
                assert seed_indices.max().item() < batch.x.shape[0]
                assert seed_indices.max().item() < batch.y.shape[0]

                seed_coords = batch.x[seed_indices, :3]

                assert batch.batch.max().item() + 1 == batch_size, \
                    f"num_subgraphs={batch.batch.max().item() + 1}, batch_size={batch_size}"
                assert batch.batch.max().item() < seed_coords.shape[0]

                nodes_center_coords = seed_coords[batch.batch]

                rel_coords = batch.x[:, :3] - nodes_center_coords
                other_features = batch.x[:, 3:]
                x_new = torch.cat([rel_coords, other_features], dim=1)

                out_target = model(x_new, batch.edge_index, target_indices=seed_indices)
                y_target = batch.y[seed_indices]

                loss = loss_fn(out_target, y_target)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

                total_loss += float(loss.item())
                total_batches += 1

            avg_train_loss = total_loss / total_batches if total_batches > 0 else 0.0

            do_validation = (epoch == 1) or (epoch % eval_every == 0)

            if do_validation:
                model.eval()
                val_loss = 0.0
                correct = 0
                total_samples = 0
                all_preds = []
                all_labels = []

                with torch.inference_mode():
                    for batch_idx, batch in enumerate(val_loader):
                        batch = batch.to(device)

                        batch_size = batch.batch_size
                        seed_indices = torch.arange(batch_size, device=device)

                        assert seed_indices.numel() == batch_size
                        assert seed_indices.max().item() < batch.x.shape[0]
                        assert seed_indices.max().item() < batch.y.shape[0]

                        seed_coords = batch.x[seed_indices, :3]

                        assert batch.batch.max().item() + 1 == batch_size, \
                            f"val num_subgraphs={batch.batch.max().item() + 1}, batch_size={batch_size}"
                        assert batch.batch.max().item() < seed_coords.shape[0]

                        nodes_center_coords = seed_coords[batch.batch]

                        rel_coords = batch.x[:, :3] - nodes_center_coords
                        other_features = batch.x[:, 3:]
                        x_new = torch.cat([rel_coords, other_features], dim=1)

                        out_target = model(x_new, batch.edge_index, target_indices=seed_indices)
                        y_target = batch.y[seed_indices]

                        if batch_idx == 0 and epoch == 1:
                            structure_ok_val = debug_check_seed_structure(
                                batch, seed_indices, y_target, phase="val", epoch=epoch, batch_idx=batch_idx
                            )
                            print("val seed_coords.shape:", seed_coords.shape)
                            print("val x_new.shape:", x_new.shape)

                        loss = loss_fn(out_target, y_target)
                        val_loss += float(loss.item())

                        pred = out_target.argmax(dim=1)
                        correct += (pred == y_target).sum().item()
                        total_samples += y_target.size(0)

                        all_preds.extend(pred.cpu().tolist())
                        all_labels.extend(y_target.cpu().tolist())

                if len(val_loader) == 0:
                    print("Validation loader is empty. Skip fold.")
                    break

                avg_val_loss = val_loss / len(val_loader)
                val_accuracy = correct / total_samples if total_samples > 0 else 0.0
                
                f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0) if all_preds else 0.0
                recall = recall_score(all_labels, all_preds, average='macro', zero_division=0) if all_preds else 0.0
                precision = precision_score(all_labels, all_preds, average='macro', zero_division=0) if all_preds else 0.0

                f1_weighted = f1_score(all_labels, all_preds, average='weighted', zero_division=0) if all_preds else 0.0
                recall_weighted = recall_score(all_labels, all_preds, average='weighted', zero_division=0) if all_preds else 0.0
                precision_weighted = precision_score(all_labels, all_preds, average='weighted', zero_division=0) if all_preds else 0.0

                scheduler.step(avg_val_loss)

            else:
                avg_val_loss = val_loss_hist[-1] if len(val_loss_hist) > 0 else np.nan
                val_accuracy = val_acc_hist[-1] if len(val_acc_hist) > 0 else np.nan
                f1 = f1_hist[-1] if len(f1_hist) > 0 else np.nan
                recall = recall_hist[-1] if len(recall_hist) > 0 else np.nan
                precision = precision_hist[-1] if len(precision_hist) > 0 else np.nan
                
                f1_weighted = f1_weighted_hist[-1] if len(f1_weighted_hist) > 0 else np.nan
                recall_weighted = recall_weighted_hist[-1] if len(recall_weighted_hist) > 0 else np.nan
                precision_weighted = precision_weighted_hist[-1] if len(precision_weighted_hist) > 0 else np.nan

            train_loss_hist.append(avg_train_loss)
            val_loss_hist.append(avg_val_loss)
            val_acc_hist.append(val_accuracy)
            
            f1_hist.append(f1)
            recall_hist.append(recall)
            precision_hist.append(precision)
            
            f1_weighted_hist.append(f1_weighted)
            recall_weighted_hist.append(recall_weighted)
            precision_weighted_hist.append(precision_weighted)

            epoch_bar.set_postfix(
                tr_loss=f"{avg_train_loss:.4f}",
                val_loss=f"{avg_val_loss:.4f}" if not np.isnan(avg_val_loss) else "skip",
                f1=f"{f1:.4f}" if not np.isnan(f1) else "skip"
            )

            if do_validation and not np.isnan(avg_val_loss):
                if avg_val_loss < best_val_loss:
                    best_val_loss = avg_val_loss
                    counter = 0
                    pth_savename = pth_dir_template.format(
                        model_name=type(model).__name__,
                        fold=fold + 1
                    )
                    torch.save(model.state_dict(), pth_savename)
                else:
                    counter += 1
                    if counter >= patience:
                        print(f"\nEarly stopping at epoch {epoch}")
                        break
        
        np.save(os.path.join(output_dir_info, f"train_loss_fold_{fold+1}.npy"), np.array(train_loss_hist))
        np.save(os.path.join(output_dir_info, f"val_loss_fold_{fold+1}.npy"), np.array(val_loss_hist))
        np.save(os.path.join(output_dir_info, f"f1_score_fold_{fold+1}.npy"), np.array(f1_hist))
        np.save(os.path.join(output_dir_info, f"recall_fold_{fold+1}.npy"), np.array(recall_hist))
        np.save(os.path.join(output_dir_info, f"precision_fold_{fold+1}.npy"), np.array(precision_hist))
        
        np.save(os.path.join(output_dir_info, f"f1_score_weighted_fold_{fold+1}.npy"), np.array(f1_weighted_hist))
        np.save(os.path.join(output_dir_info, f"recall_weighted_fold_{fold+1}.npy"), np.array(recall_weighted_hist))
        np.save(os.path.join(output_dir_info, f"precision_weighted_fold_{fold+1}.npy"), np.array(precision_weighted_hist))

        all_fold_metrics.append({
            "fold": fold + 1,
            "best_val_loss": best_val_loss,
            "final_val_accuracy": val_accuracy if len(val_acc_hist) > 0 else np.nan,
        })

        print(f"Completed Fold {fold + 1}")
        print(f"structure_ok_train: {structure_ok_train}")
        print(f"structure_ok_val: {structure_ok_val}")

    pd.DataFrame(all_fold_metrics).to_csv(output_dir_csv, index=False)

print("All datasets processed.")